# Molecular fingerprint generation

Source: archived main notebook, cells 42–61. Set the external conformer directory for 3D fingerprints; pretrained maps are loaded only when present.


In [ ]:
from pathlib import Path
import os
import sys

def repository_root(start):
    for path in (start, *start.parents):
        if (path / "Code" / "model_train.py").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the CycloAddGenerator repository.")

REPO_ROOT = repository_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks" / "03_modeling"))
DATA_DIR = REPO_ROOT / "Data" / "DFT_Result"
DESCRIPTOR_MAP = REPO_ROOT / "Data" / "Descriptors" / "Descriptors_.pkl"

In [ ]:
import pickle
import numpy as np
from Code import gendes, model_train
from _modeling_common import load_aligned

data_Ga, X_Ga = load_aligned(DATA_DIR / "Active_Energy.csv", DESCRIPTOR_MAP)
OUTPUT_DIR = DATA_DIR / "Descriptors"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MOL_DIR = None  # External directory containing smilesid_*.mol files.
PRETRAINED = {
    "VAE": OUTPUT_DIR / "smiles_vae_map.pkl",
    "Transformer": OUTPUT_DIR / "smiles_trfm_map.pkl",
    "Graph autoencoder": OUTPUT_DIR / "graph_gae_map.pkl",
}

## 2D fingerprints and descriptors


In [ ]:
generator_2d = gendes.generate2Ddesc(data_Ga, has_product=1)
descriptor_maps = {
    "RDKit fingerprint": generator_2d.calc_rdkit_mf(),
    "RDKit descriptors": generator_2d.calc_rdkit_desc(),
    "Morgan fingerprint": generator_2d.calc_morgan_mf(),
    "Mordred descriptors": generator_2d.calc_modred_desc(),
}

## Optional 3D descriptors


In [ ]:
if MOL_DIR is not None:
    if not Path(MOL_DIR).is_dir():
        raise FileNotFoundError(MOL_DIR)
    generator_3d = gendes.generate3Ddesc(data_Ga, mol_dir=str(MOL_DIR))
    descriptor_maps.update({
        "ACSF": generator_3d.calc_acsf_desc(),
        "SOAP": generator_3d.calc_soap_desc(),
        "MBTR": generator_3d.calc_mbtr_desc(),
        "LMBTR": generator_3d.calc_lmbtr_desc(),
    })
else:
    print("3D descriptors skipped: set MOL_DIR to enable them.")

## Optional pretrained representations


In [ ]:
for name, path in PRETRAINED.items():
    if path.is_file():
        with path.open("rb") as handle:
            descriptor_maps[name] = pickle.load(handle)
    else:
        print(f"Skipping {name}: {path} is absent")

## Align and save the benchmark bundle


In [ ]:
fingerprints = {"PhysOrg + SVO": X_Ga}
for name, mapping in descriptor_maps.items():
    matrix, _, retained = model_train.descriptor_generator(
        [data_Ga], mapping, mf_des=True, with_product=False,
        y_name=None, is_3d_map=False)
    if matrix.shape[0] != len(data_Ga) or retained != list(range(len(data_Ga))):
        raise ValueError(f"Representation {name} does not cover all aligned rows")
    fingerprints[name] = matrix
bundle = {
    "features": fingerprints,
    "targets": {column: data_Ga[column].to_numpy()
                for column in ["Diene_Distort", "Ene_Distort",
                               "Interaction", "deltaGa(Solvent)"]},
    "reaction_keys": list(zip(data_Ga["Diene_Index"].astype(int),
                              data_Ga["Ene_Index"].astype(int),
                              data_Ga["Structure_id"].astype(int))),
}
bundle_path = OUTPUT_DIR / "ZINC_fingerprint_bundle.pkl"
with bundle_path.open("wb") as handle:
    pickle.dump(bundle, handle, protocol=pickle.HIGHEST_PROTOCOL)
print(f"Saved {len(fingerprints)} representations to {bundle_path}")